In [164]:
# %load_ext autoreload
# %autoreload 2

In [165]:
import pandas as pd

In [166]:
from my_shelves.utils.dataset import get_books, get_reviews

In [167]:
books_df = get_books(data_dir="../data")
reviews_df = get_reviews(data_dir="../data")

In [168]:
books_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,NaN,7,['189911'],US,eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",B00071IKUY,False,4.03,NaN,...,NaN,Book Club Edition,1987.0,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140,8948723,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,0743294297,3282,[],US,eng,"[{'count': '7615', 'name': 'to-read'}, {'count...",NaN,False,3.49,B002ENBLOK,...,7.0,NaN,2009.0,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...,6066819,51184,6243154,Best Friends Forever,Best Friends Forever
2,NaN,60,['1052227'],US,eng,"[{'count': '54', 'name': 'currently-reading'},...",B01NCIKAQX,True,4.33,B01NCIKAQX,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,33394837,269,54143148,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2)
3,555118000X,19,[],US,eng,"[{'count': '3488', 'name': 'to-read'}, {'count...",NaN,False,3.82,NaN,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/89373.The_...,https://s.gr-assets.com/assets/nophoto/book/11...,89373,77,1080201,The Bonfire of the Vanities,The Bonfire of the Vanities
4,0842379428,566,[],US,eng,"[{'count': '6393', 'name': 'to-read'}, {'count...",NaN,False,4.26,B000FCKCJC,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/89376.Heaven,https://images.gr-assets.com/books/1406508230m...,89376,7345,86257,Heaven,Heaven


In [169]:
reviews_df.head()

,user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments
0,8842281e1d1347389f2ab93d60773d4d,19398490,ea4a220b10e6b5c796dae0e3b970aff1,4,A beautiful story. It is rare to encounter a b...,Sun Jan 03 21:20:46 -0800 2016,Tue Sep 20 23:30:15 -0700 2016,Tue Sep 13 11:51:51 -0700 2016,Sat Aug 20 07:03:03 -0700 2016,35,5
1,8842281e1d1347389f2ab93d60773d4d,8664353,da2d4cfce836a2c57ad55c38437aa692,5,"Wow. Amazing story, and well told - kept me up...",Sun Nov 20 09:10:15 -0800 2011,Wed Mar 22 11:47:04 -0700 2017,Wed May 16 00:00:00 -0700 2012,Sun May 06 00:00:00 -0700 2012,50,4
2,01ec1a320ffded6b2dd47833f2c8e4fb,27878417,b22dd337ee1938dd7cac80261aed6d2f,5,Contains major spoilers so only read this afte...,Mon Nov 16 21:35:17 -0800 2015,Sun Dec 06 22:45:19 -0800 2015,Wed Nov 18 11:06:15 -0800 2015,Mon Nov 16 00:00:00 -0800 2015,16,0
3,01ec1a320ffded6b2dd47833f2c8e4fb,23518092,39afe8c459a61c9d66fe212217608d72,4,"This is a juicy, fast-paced and totally enjoya...",Sun Jan 18 22:08:52 -0800 2015,Sun Jan 18 22:09:49 -0800 2015,Sun Jan 18 00:00:00 -0800 2015,NaN,6,2
4,01ec1a320ffded6b2dd47833f2c8e4fb,23350137,544dca1efd8819f93996a20fec44f199,4,"This is a fun, loving, sexy and very satisfyin...",Sun Nov 30 23:05:08 -0800 2014,Tue Dec 02 08:25:58 -0800 2014,Mon Dec 01 00:00:00 -0800 2014,Sun Nov 30 00:00:00 -0800 2014,6,0


In [170]:
reviews_df.shape

(110000, 11)

In [171]:
def clean_reviews_text(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    df = df.dropna(subset=[column_name])
    df = df[df[column_name].str.strip() != ""]
    return df

In [172]:
reviews_df = clean_reviews_text(reviews_df, column_name="review_text")
reviews_df.shape

(109963, 11)

In [173]:
def add_reviews_count_words(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    df[f"{column_name}_n_words"] = df[column_name].apply(lambda x: len(x.split()))
    return df

In [174]:
reviews_df = add_reviews_count_words(reviews_df, column_name="review_text")
reviews_df.shape

(109963, 12)

In [175]:
def clean_small_reviews(df: pd.DataFrame, column_name: str, min_words: int) -> pd.DataFrame:
    df = df[df[f"{column_name}_n_words"] > min_words]
    return df

In [176]:
reviews_df = clean_small_reviews(reviews_df, column_name="review_text", min_words=50)

Remove useless columns

In [177]:
def convert_column_to_datetime(df: pd.DataFrame, column_name: str, replacement_value: str="futur") -> pd.DataFrame:
    replacement_date = 'Sun Jul 1 00:00:00 -0700 1970'
    if replacement_value == "futur":
        replacement_date = 'Sun Jul 1 00:00:00 -0700 2050'
    df[column_name] = pd.to_datetime(df[column_name],
                                     format='%a %b %d %H:%M:%S %z %Y',
                                     utc=True,
                                     errors='coerce')
    df[column_name] = df[column_name].fillna(value=replacement_date)
    return df

In [178]:
def clean_reviews_dates(df: pd.DataFrame) -> pd.DataFrame:
    date_columns = {"started_at": "past",
                    "read_at": "futur",
                    "date_added": "futur",
                    "date_updated": "futur"}
    for column, replacement_value in date_columns.items():
        df = convert_column_to_datetime(df,
                                        column_name=column,
                                        replacement_value=replacement_value)
    return df

In [179]:
reviews_df = clean_reviews_dates(reviews_df)

In [180]:
def compute_read_duration(df: pd.DataFrame, start_column: str, end_column: str) -> pd.DataFrame:
    df["read_duration"] = (df[end_column] - df[start_column]).dt.total_seconds() / 3600
    return df

In [181]:
reviews_df = compute_read_duration(reviews_df, start_column="started_at", end_column="read_at")
reviews_df.head()

,user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments,review_text_n_words,read_duration
0,8842281e1d1347389f2ab93d60773d4d,19398490,ea4a220b10e6b5c796dae0e3b970aff1,4,A beautiful story. It is rare to encounter a b...,2016-01-04 05:20:46+00:00,2016-09-21 06:30:15+00:00,2016-09-13 18:51:51+00:00,2016-08-20 14:03:03+00:00,35,5,258,580.813333
1,8842281e1d1347389f2ab93d60773d4d,8664353,da2d4cfce836a2c57ad55c38437aa692,5,"Wow. Amazing story, and well told - kept me up...",2011-11-20 17:10:15+00:00,2017-03-22 18:47:04+00:00,2012-05-16 07:00:00+00:00,2012-05-06 07:00:00+00:00,50,4,264,240.000000
3,01ec1a320ffded6b2dd47833f2c8e4fb,23518092,39afe8c459a61c9d66fe212217608d72,4,"This is a juicy, fast-paced and totally enjoya...",2015-01-19 06:08:52+00:00,2015-01-19 06:09:49+00:00,2015-01-18 08:00:00+00:00,1970-07-01 07:00:00+00:00,6,2,144,390529.000000
4,01ec1a320ffded6b2dd47833f2c8e4fb,23350137,544dca1efd8819f93996a20fec44f199,4,"This is a fun, loving, sexy and very satisfyin...",2014-12-01 07:05:08+00:00,2014-12-02 16:25:58+00:00,2014-12-01 08:00:00+00:00,2014-11-30 08:00:00+00:00,6,0,216,24.000000
5,01ec1a320ffded6b2dd47833f2c8e4fb,22911904,1427adba9fafea91142088973a49927b,4,I loved this book!! Pure enjoyment!! Love the ...,2014-08-13 20:42:16+00:00,2014-11-10 16:15:42+00:00,2014-11-09 08:00:00+00:00,2014-11-09 08:00:00+00:00,3,3,305,0.000000


In [36]:
reviews_df.shape

(62289, 13)

In [ ]:
cols_to_drop = ["user_id", "date_updated"]

In [1]:
import os
os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

'/home/chris/code/christophezito/gcp/my-shelves-493916-4fc17c3c5ea6.json'

In [183]:
from my_shelves.utils.params import GCP_PROJECT, BQ_DATASET
from my_shelves.utils.bigquery import get_book

df = get_book(22077083)
df


/home/chris/.pyenv/versions/my_shelves_env/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,None,1,['161354'],US,eng,"[{'count': '559', 'name': 'to-read'}, {'count'...",None,False,3.8,None,...,NaN,None,1992.0,https://www.goodreads.com/book/show/22077083-t...,https://images.gr-assets.com/books/1399953592m...,22077083,5,2232945,The Haunted Fort (The Hardy Boys # 03),The Haunted Fort (The Hardy Boys # 03)


In [17]:
import pandas as pd

In [25]:
reviews_clean_df = pd.read_csv("../data/reviews_cleaned.csv")
reviews_clean_df.head()

,book_id,review_text,rating,n_votes,read_duration
0,1362,['Did you know that Egyptian women urinate sta...,4.103448,35,366.249693
1,1365,['It may be surprising to find out that one of...,4.000000,2,500.532870
2,1366,"[""I'm a historian and this is the book that st...",4.400000,10,4018.185778
3,1367,"[""Refreshingly amusing. In the histories, Hero...",4.000000,1,2049.278056
4,1368,"['Herodotus was hailed as ""The Father of Histo...",5.000000,6,4992.000000


In [26]:
reviews_clean_df.shape

(6879, 5)

In [22]:
df_grouped = reviews_clean_df.groupby("book_id").agg({
    "review_text": list,
    "rating": "mean",
    "n_votes": "sum",
    "read_duration": "mean"
}).reset_index()

In [23]:
df_grouped.head()

,book_id,review_text,rating,n_votes,read_duration
0,1362,[Did you know that Egyptian women urinate stan...,3.0,3,0.0
1,5345,[Ho-hum. Not that thrilling. Non-fiction book ...,2.0,0,0.0
2,5346,[I loved John Grisham when he first hit the li...,4.0,0,216.0
3,13872,"[I had to look up ""geek"" because it surely did...",4.0,4,24.0
4,14069,[Sent from China on a journey to Istanbul to r...,4.0,0,0.0


In [2]:
import pandas as pd

In [5]:
pd.read_csv("../data/reviews_ENG_mini.csv").shape

(110000, 11)

In [7]:
pd.read_csv("../data/goodreads_reviews_ENG_10k.csv").shape

(10000, 11)

In [8]:
pd.read_csv("../data/goodreads_reviews_ENG_100k.csv").shape

(100000, 11)

In [13]:
pd.read_csv("../data/goodreads_reviews_ENG_1M.csv").shape

(1000000, 11)

In [4]:
pd.read_csv("../data/goodreads_reviews_ENG_all.csv").shape

(11409758, 11)

## Cleaned Reviews

In [16]:
pd.read_csv("../data/reviews_cleaned_ENG_10k.csv").shape

(5019, 5)

In [17]:
pd.read_csv("../data/reviews_cleaned_ENG_100k.csv").shape

(37774, 5)

In [18]:
pd.read_csv("../data/reviews_cleaned_ENG_1M.csv").shape

(189563, 5)

In [19]:
reviews_clean_1m_df = pd.read_csv("../data/reviews_cleaned_ENG_1M.csv")
reviews_clean_1m_df.head()

,book_id,review_text,rating,n_votes,read_duration
0,1,"[""First of all, Im in love with these books si...",4.487500,34,176.873063
1,2,"[""Rowling really hits it out of the park with ...",4.388350,243,793.423830
2,3,"[""I remember trying 3 times to read this but I...",4.423729,259,435.510829
3,4,"[""I actually read this book first, thinking it...",5.000000,4,258.948750
4,5,"[""Rowling does it again with Harry and the gan...",4.636364,79,744.728725


In [5]:
reviews_clean_all_df = pd.read_csv("../data/reviews_cleaned_ENG_all.csv")
reviews_clean_all_df.head()

,book_id,review_text,rating,n_votes,read_duration
0,1,"[""First of all, Im in love with these books si...",4.487402,1288,348.994088
1,2,"[""Rowling really hits it out of the park with ...",4.328467,2233,621.952282
2,3,"[""I remember trying 3 times to read this but I...",4.437788,4964,414.159174
3,4,"[""I actually read this book first, thinking it...",4.529412,80,156.405964
4,5,"[""Rowling does it again with Harry and the gan...",4.582845,2022,320.308168


In [6]:
reviews_clean_all_df.shape

(655288, 5)

## Books dataset

In [16]:
books_df_10k = pd.read_csv("../data/raw/goodreads_books_ENG.csv")
books_df_10k.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,NaN,7.0,['189911'],US,eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",B00071IKUY,False,4.03,NaN,...,NaN,Book Club Edition,1987.0,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140.0,8948723.0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,0743294297,3282.0,[],US,eng,"[{'count': '7615', 'name': 'to-read'}, {'count...",NaN,False,3.49,B002ENBLOK,...,7.0,NaN,2009.0,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...,6066819,51184.0,6243154.0,Best Friends Forever,Best Friends Forever
2,NaN,60.0,['1052227'],US,eng,"[{'count': '54', 'name': 'currently-reading'},...",B01NCIKAQX,True,4.33,B01NCIKAQX,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,33394837,269.0,54143148.0,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2)
3,555118000X,19.0,[],US,eng,"[{'count': '3488', 'name': 'to-read'}, {'count...",NaN,False,3.82,NaN,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/89373.The_...,https://s.gr-assets.com/assets/nophoto/book/11...,89373,77.0,1080201.0,The Bonfire of the Vanities,The Bonfire of the Vanities
4,0842379428,566.0,[],US,eng,"[{'count': '6393', 'name': 'to-read'}, {'count...",NaN,False,4.26,B000FCKCJC,...,NaN,NaN,NaN,https://www.goodreads.com/book/show/89376.Heaven,https://images.gr-assets.com/books/1406508230m...,89376,7345.0,86257.0,Heaven,Heaven


In [ ]:
features_to_keep = ['book_id','description','publication_year','image_url','url','title','num_pages','series', 'ratings_count']

## Extended Reviews

In [25]:
reviews_clean_10k_df = pd.read_csv("../data/reviews_cleaned_ENG_10k.csv")
reviews_clean_10k_df.head()

,book_id,review_text,rating,n_votes,read_duration
0,2,"[""Rowling really hits it out of the park with ...",5.0,0,1512.0000
1,3,"[""I remember trying 3 times to read this but I...",4.0,0,43.7325
2,11,"['Good book for when you are sick in bed, or o...",3.0,0,0.0000
3,21,"[""A fascinating history of science. Ever curio...",3.0,46,0.0000
4,27,['Bryson is my kind of travel writer.He rarely...,4.5,0,84.0000


In [26]:
reviews_extend_10k_df = pd.read_csv("../data/reviews_extended_ENG_10k.csv")
reviews_extend_10k_df.head()

,book_id,review_text,rating,n_votes,read_duration,city,type,country,continent,province
0,2,"[""Rowling really hits it out of the park with ...",5.0,0,1512.0000,b'North Judithbury',b'mountain',b'Peru',b'Americas',b'Amazonas'
1,3,"[""I remember trying 3 times to read this but I...",4.0,0,43.7325,b'East Jill',b'forest',b'Saint Vincent and the Grenadines',b'Americas',b'Grenadines'
2,11,"['Good book for when you are sick in bed, or o...",3.0,0,0.0000,b'New Roberttown',b'beach',b'Cayman Islands',b'Americas',b'Eastern'
3,21,"[""A fascinating history of science. Ever curio...",3.0,46,0.0000,b'East Jessetown',b'mountain',b'Saint Kitts and Nevis',b'Americas',b'Saint Paul Charlestown'
4,27,['Bryson is my kind of travel writer.He rarely...,4.5,0,84.0000,b'Lake Debra',b'mountain',b'Guadeloupe',b'Americas',b'Basse-Terre'


In [28]:
reviews_clean_10k_df.shape

(5019, 5)

In [27]:
reviews_extend_10k_df.shape

(5019, 10)

In [7]:
reviews_extend_100k_df = pd.read_csv("../data/reviews_extended_ENG_100k.csv")
reviews_extend_100k_df.head()

,book_id,review_text,rating,n_votes,read_duration,city,type,country,continent,province
0,1,"[""First of all, Im in love with these books si...",4.714286,13,69.431071,b'North Judithbury',b'mountain',b'Peru',b'Americas',b'Amazonas'
1,2,"[""Rowling really hits it out of the park with ...",4.470588,31,393.387778,b'East Jill',b'forest',b'Saint Vincent and the Grenadines',b'Americas',b'Grenadines'
2,3,"[""I remember trying 3 times to read this but I...",4.210526,32,75.433523,b'New Roberttown',b'beach',b'Cayman Islands',b'Americas',b'Eastern'
3,4,"[""I actually read this book first, thinking it...",5.000000,0,0.000000,b'East Jessetown',b'mountain',b'Saint Kitts and Nevis',b'Americas',b'Saint Paul Charlestown'
4,5,"[""Rowling does it again with Harry and the gan...",4.666667,4,219.603542,b'Lake Debra',b'mountain',b'Guadeloupe',b'Americas',b'Basse-Terre'


In [10]:
reviews_extend_100k_df.shape

(37774, 10)

In [11]:
reviews_extend_1M_df = pd.read_csv("../data/reviews_extended_ENG_1M.csv")
reviews_extend_1M_df.head()

,book_id,review_text,rating,n_votes,read_duration,city,type,country,continent,province
0,1,"[""First of all, Im in love with these books si...",4.487500,34,176.873063,b'North Judithbury',b'mountain',b'Peru',b'Americas',b'Amazonas'
1,2,"[""Rowling really hits it out of the park with ...",4.388350,243,793.423830,b'East Jill',b'forest',b'Saint Vincent and the Grenadines',b'Americas',b'Grenadines'
2,3,"[""I remember trying 3 times to read this but I...",4.423729,259,435.510829,b'New Roberttown',b'beach',b'Cayman Islands',b'Americas',b'Eastern'
3,4,"[""I actually read this book first, thinking it...",5.000000,4,258.948750,b'East Jessetown',b'mountain',b'Saint Kitts and Nevis',b'Americas',b'Saint Paul Charlestown'
4,5,"[""Rowling does it again with Harry and the gan...",4.636364,79,744.728725,b'Lake Debra',b'mountain',b'Guadeloupe',b'Americas',b'Basse-Terre'


In [12]:
reviews_extend_1M_df.shape

(189563, 10)

In [13]:
reviews_extend_all_df = pd.read_csv("../data/reviews_extended_ENG_all.csv")
reviews_extend_all_df.head()

,book_id,review_text,rating,n_votes,read_duration,city,type,country,continent,province
0,1,"[""First of all, Im in love with these books si...",4.487402,1288,348.994088,b'North Judithbury',b'mountain',b'Peru',b'Americas',b'Amazonas'
1,2,"[""Rowling really hits it out of the park with ...",4.328467,2233,621.952282,b'East Jill',b'forest',b'Saint Vincent and the Grenadines',b'Americas',b'Grenadines'
2,3,"[""I remember trying 3 times to read this but I...",4.437788,4964,414.159174,b'New Roberttown',b'beach',b'Cayman Islands',b'Americas',b'Eastern'
3,4,"[""I actually read this book first, thinking it...",4.529412,80,156.405964,b'East Jessetown',b'mountain',b'Saint Kitts and Nevis',b'Americas',b'Saint Paul Charlestown'
4,5,"[""Rowling does it again with Harry and the gan...",4.582845,2022,320.308168,b'Lake Debra',b'mountain',b'Guadeloupe',b'Americas',b'Basse-Terre'


In [14]:
reviews_extend_all_df.shape

(655288, 10)